# RSS Vector Retriever Manual Test

Use this notebook to run the `RSSVectorRetrieverAgent` against your MongoDB Atlas data.

**Prerequisites**
- `.env` has valid Mongo credentials and (optionally) `OPENAI_API_KEY`.
- `rss_feeds` is populated and the ingestion worker has produced embedded documents in `rss_items`.

Adjust the query cell below and rerun to inspect different results.

In [ ]:
# Environment and imports
import sys
from datetime import datetime

sys.path.append('..')

from app.config import settings
from app.db.client import connect_to_mongo, close_mongo_connection, mongo_client
from app.agents.state import create_initial_state, SearchQuery
from app.agents.rss_vector_retriever_agent import RSSVectorRetrieverAgent

print(f'Environment: {settings.ENVIRONMENT}')
print(f'Embeddings provider: {settings.EMBEDDINGS_PROVIDER}')

In [ ]:
# Connect to MongoDB
await connect_to_mongo()
collections = await mongo_client.database.list_collection_names()
print('Connected. Collections:', collections)

In [ ]:
# Configure and run the agent
QUERY = 'best gaming laptop deals'
MAX_RESULTS = 5
FRESHNESS_DAYS = 14

state = create_initial_state(QUERY)
state['search_query'] = SearchQuery(raw_query=QUERY, normalized_query=QUERY)
agent = RSSVectorRetrieverAgent(max_results=MAX_RESULTS, freshness_days=FRESHNESS_DAYS)
state = await agent.process(state)
last_step = state['agent_steps'][-1] if state['agent_steps'] else None
print('Agent status:', last_step.status if last_step else 'unknown')
print('RSS items returned:', len(state['rss_results']))

In [ ]:
# Inspect results
from pprint import pprint

for idx, item in enumerate(state['rss_results'], start=1):
    title = item.get('title') or 'Untitled'
    print(f"#{idx} {title}")
    pprint({k: v for k, v in item.items() if k != 'title'})
    print('-' * 60)

In [ ]:
# Close Mongo connection when finished
await close_mongo_connection()
print('Mongo connection closed.')